Load the preprocessed data from notebook 02 and split back into train/test.

In [13]:
from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
df = pd.read_csv(DATA_DIR / "processed/sales_preprocessed.csv")

train = df[df["split"] == "train"].drop(columns=["split"])
test = df[df["split"] == "test"].drop(columns=["split"])

y_train = train["ClosePrice"]
X_train = train.drop(columns=["ClosePrice"])
y_test = test["ClosePrice"]
X_test = test.drop(columns=["ClosePrice"])

print(X_train.shape, X_test.shape)

(122623, 551) (10158, 551)


In [14]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
print("train R^2:", model.score(X_train, y_train))

train R^2: 0.7269387938372315


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("test MAE:", mae)
print("test RMSE:", rmse)
print("test R^2:", r2)

test MAE: 276304.03550222534
test RMSE: 434921.3235080184
test R^2: 0.7340468775354496


In [17]:
import numpy as np

y_train_log = np.log(y_train)

In [18]:
from sklearn.linear_model import LinearRegression

model_log = LinearRegression()
model_log.fit(X_train, y_train_log)
print("train R^2 (log price):", model_log.score(X_train, y_train_log))

train R^2 (log price): 0.8385810935754356


In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_test_log = np.log(y_test)
y_pred_log = model_log.predict(X_test)
y_pred_from_log = np.exp(y_pred_log)

print("test R^2 (log price):", r2_score(y_test_log, y_pred_log))
print("test MAE (dollars):", mean_absolute_error(y_test, y_pred_from_log))
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test, y_pred_from_log)))
print("test R^2 (dollars):", r2_score(y_test, y_pred_from_log))

test R^2 (log price): 0.8378194312225306
test MAE (dollars): 231111.26288126246
test RMSE (dollars): 423751.74302872695
test R^2 (dollars): 0.7475317981524725


Regression with 2 year train split instead of 1 year

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
df2 = pd.read_csv(DATA_DIR / "processed/sales_preprocessed_2.csv")

train2 = df2[df2["split"] == "train"].drop(columns=["split"])
test2 = df2[df2["split"] == "test"].drop(columns=["split"])

y_train2 = train2["ClosePrice"]
X_train2 = train2.drop(columns=["ClosePrice"])
y_test2 = test2["ClosePrice"]
X_test2 = test2.drop(columns=["ClosePrice"])
print("2yr shapes:", X_train2.shape, X_test2.shape)

model2 = LinearRegression()
model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)
print("\nraw price")
print("train R^2:", model2.score(X_train2, y_train2))
print("test RMSE:", np.sqrt(mean_squared_error(y_test2, y_pred2)))
print("test R^2:", r2_score(y_test2, y_pred2))

y_train2_log = np.log(y_train2)
y_test2_log = np.log(y_test2)
model2_log = LinearRegression()
model2_log.fit(X_train2, y_train2_log)
y_pred2_from_log = np.exp(model2_log.predict(X_test2))
print("\nwith log scaling")
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test2, y_pred2_from_log)))
print("test R^2 (dollars):", r2_score(y_test2, y_pred2_from_log))

2yr shapes: (241678, 601) (9849, 601)

raw price
train R^2: 0.7306764520408057
test RMSE: 433750.54800552333
test R^2: 0.7332620223872831


In [25]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=12, min_samples_leaf=50, random_state=0)
tree.fit(X_train2, y_train2_log)

y_pred_tree_log = tree.predict(X_test2)
y_pred_tree = np.exp(y_pred_tree_log)

print("decision tree (log price target)")
print("train R^2 (log):", tree.score(X_train2, y_train2_log))
print("test R^2 (log):", r2_score(y_test2_log, y_pred_tree_log))
print("test RMSE (dollars):", np.sqrt(mean_squared_error(y_test2, y_pred_tree)))
print("test R^2 (dollars):", r2_score(y_test2, y_pred_tree))

decision tree (log price target)
train R^2 (log): 0.8613409956631988
test R^2 (log): 0.853450408985302
test RMSE (dollars): 396929.84331410914
test R^2 (dollars): 0.7766261721734594
